In [1]:
 %load_ext autoreload
%autoreload 2
import numpy as np
from utils_final_results import load_dataset
from hydra import compose
from sklearn.linear_model import Lasso, LassoCV
from sklearn.metrics import mean_squared_error
from utils_table_generator import get_mean_std, flatten_config
from omegaconf import OmegaConf
import pandas as pd

/home/lcornelis/anaconda3/envs/proteo/lib/python3.11/site-packages/torch_geometric/typing.py:155: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: /home/lcornelis/anaconda3/envs/proteo/lib/python3.11/site-packages/torch_spline_conv/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(
/home/lcornelis/code/TopoProteo/tutorials/utils_final_results.py:55: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  initialize(config_path="../configs", job_name="job")


In [2]:
def construct_dataset(dataset):
    X_list = []
    y_list = []
    sex_list = []
    mutation_list = []
    age_list = []

    for data in dataset:
        graph_feat = data.x.view(-1).numpy()
        X_list.append(graph_feat)
        y_list.append(data.y.item())
        sex_list.append(data.sex.item())
        mutation_list.append(data.mutation.item())
        age_list.append(data.age.item())

    X = np.vstack(X_list)
    y = np.array(y_list)

    sex = np.array(sex_list).reshape(-1, 1)
    mutation = np.array(mutation_list).reshape(-1, 1)
    age = np.array(age_list).reshape(-1, 1)

    return X, y, sex, mutation, age


def stack_features(
    train_features, train_labels, train_sex, train_mutation, train_age,
    val_features, val_labels, val_sex, val_mutation, val_age,
    test_features=None, test_labels=None, test_sex=None, test_mutation=None, test_age=None
):
    target_shape = train_features.shape[1] // 3
    if train_sex.shape[1] < target_shape:
        # Calculate how many times to repeat each covariate along the second axis
        repeat_times = target_shape // train_sex.shape[1]
        train_sex = np.tile(train_sex, (1, repeat_times))
        val_sex = np.tile(val_sex, (1, repeat_times))
        train_mutation = np.tile(train_mutation, (1, repeat_times))
        val_mutation = np.tile(val_mutation, (1, repeat_times))
        train_age = np.tile(train_age, (1, repeat_times))
        val_age = np.tile(val_age, (1, repeat_times))
        # If test features are provided, tile them as well
        if test_sex is not None:
            test_sex = np.tile(test_sex, (1, repeat_times))
        if test_mutation is not None:
            test_mutation = np.tile(test_mutation, (1, repeat_times))
        if test_age is not None:
            test_age = np.tile(test_age, (1, repeat_times))

    train_combined = np.hstack((train_features, train_sex, train_mutation, train_age))
    print(train_combined.shape)
    val_combined = np.hstack((val_features, val_sex, val_mutation, val_age))
    print(val_combined.shape)
    # If test features are provided, process and return them as well
    if test_features is not None and test_sex is not None and test_mutation is not None and test_age is not None:
        test_combined = np.hstack((test_features, test_sex, test_mutation, test_age))
        print(test_combined.shape)
        return train_combined, val_combined, test_combined, train_labels, val_labels, test_labels
    else:
        return train_combined, val_combined, train_labels, val_labels

def concat_features(
    train_features, train_labels, train_sex, train_mutation, train_age,
    val_features, val_labels, val_sex, val_mutation, val_age,
    test_features=None, test_labels=None, test_sex=None, test_mutation=None, test_age=None
):
    train_combined = np.hstack((train_features, train_sex, train_mutation, train_age))
    val_combined = np.hstack((val_features, val_sex, val_mutation, val_age))
    # If test features are provided, process and return them as well
    if test_features is not None and test_sex is not None and test_mutation is not None and test_age is not None:
        test_combined = np.hstack((test_features, test_sex, test_mutation, test_age))
        return train_combined, val_combined, test_combined, train_labels, val_labels, test_labels
    else:
        return train_combined, val_combined, train_labels, val_labels

def convert_results(norm_mean, adj_metric, adj_thresh, k_fold, num_folds, fold) :
    cfg = compose(
        config_name="run.yaml",
        overrides=[
            "model=graph/gcn",
            "dataset=graph/FTD",
            f"dataset.loader.parameters.adj_metric={adj_metric}",
            f"dataset.loader.parameters.adj_thresh={adj_thresh}",
            f"dataset.loader.parameters.kfold={k_fold}",
            f"dataset.loader.parameters.num_folds={num_folds}",
            f"dataset.loader.parameters.fold={fold}",
            "dataset.loader.parameters.y_val=global_cog_slope",
            "dataset.loader.parameters.two_pass=True",
            "dataset.loader.parameters.num_nodes=3667",
        ],
        return_hydra_config=True
    )
    cfg_dict = OmegaConf.to_container(cfg, resolve=False)
    cfg_flat = flatten_config(cfg_dict)
    mean, std = get_mean_std(cfg_flat)
    assert mean is not None, "Mean is None for run"
    assert std is not None, "Std is None for run"
    orig_rmse = np.sqrt(norm_mean) * std
    # orig_std = norm_std * std**2
    return orig_rmse


def run_lasso_folds(
    num_folds=5,
    k_fold=True,
    use_features=True,
    adj_metric="wgcna",
    adj_thresh=0.5, 
):
    """
    If k_fold=True:
        - Perform per-fold tuning of alpha using train -> val.
        - Aggregate val MSE across folds.
        - Return alpha-wise results and best_alpha_global.

    If k_fold=False:
        - Use a single train/val/test split (fold=0).
        - Tune alpha using train -> val.
        - Fit best-alpha model on TRAIN only.
        - Report TRAIN performance as final.
    """

    alphas = np.logspace(-4, 2, 50)

    # ===============================
    # K-FOLD CROSS-VALIDATION MODE
    # ===============================
    if k_fold:
        alpha_results = {alpha: [] for alpha in alphas}
        alpha_results_orig = {alpha: [] for alpha in alphas}

        for fold in range(num_folds):
            train, val, test = load_dataset(
                adj_metric,
                adj_thresh,
                k_fold,
                num_folds,
                fold=fold,
                y_val="global_cog_slope",
                two_pass=True,
                num_nodes=3667,
            )

            train_features, train_labels, train_sex, train_mutation, train_age = construct_dataset(train)
            val_features, val_labels, val_sex, val_mutation, val_age = construct_dataset(val)

            if use_features:
                X_train, X_val, y_train, y_val = stack_features(
                    train_features, train_labels, train_sex, train_mutation, train_age,
                    val_features, val_labels, val_sex, val_mutation, val_age,
                )
            else:
                # Only use original train_features, no sex/mutation/age
                X_train, X_val = train_features, val_features
                y_train, y_val = train_labels, val_labels

            print(f"Fold {fold}: Training on {X_train.shape[0]} samples, validating on {X_val.shape[0]} samples")

            best_alpha_fold = None
            best_mse_fold = float("inf")

            for alpha in alphas:
                model = Lasso(alpha=alpha, random_state=42, max_iter=5000)
                model.fit(X_train, y_train)

                preds = model.predict(X_val)
                mse = mean_squared_error(y_val, preds)
                mse_orig = convert_results(mse, adj_metric, adj_thresh, True, num_folds, fold)

                alpha_results[alpha].append(mse)
                alpha_results_orig[alpha].append(mse_orig)

                if mse < best_mse_fold:
                    best_mse_fold = mse
                    best_alpha_fold = alpha

            # Optional: fit best model per fold on train (not used for test here)
            best_model_fold = Lasso(alpha=best_alpha_fold, random_state=42, max_iter=5000)
            best_model_fold.fit(X_train, y_train)

            print(f"Completed fold {fold}: best_alpha={best_alpha_fold:.5f}, val_MSE={best_mse_fold:.4f}")

        # Aggregate across folds
        print("\nValidation results for each alpha value across folds:")
        print("Alpha\tMean MSE ± Std\tMean(orig) ± Std(orig)")
        print("-" * 40)

        best_mean_mse = float("inf")
        best_alpha_global = None

        for alpha in alphas:
            vals = alpha_results[alpha]
            if not vals:
                continue
            mean_mse = np.mean(vals)
            std_mse = np.std(vals)
            mean_orig = np.mean(alpha_results_orig[alpha])
            std_orig = np.std(alpha_results_orig[alpha])

            print(f"{alpha:.4f}\t{mean_mse:.4f} ± {std_mse:.4f}\t{mean_orig:.4f} ± {std_orig:.4f}")

            if mean_mse < best_mean_mse:
                best_mean_mse = mean_mse
                best_alpha_global = alpha

        print(f"\nBest alpha across folds (by val MSE): {best_alpha_global} (mean MSE={best_mean_mse:.4f})")

        return {
            "mode": "cv",
            "alpha_val_mse": alpha_results,
            "alpha_val_mse_orig": alpha_results_orig,
            "best_alpha_global": best_alpha_global,
        }

    # ===============================
    # SINGLE-SPLIT MODE (k_fold=False)
    # ===============================
    else:
        # One deterministic split; assumes load_dataset returns train/val/test
        train, val, test = load_dataset(
            adj_metric,
            adj_thresh,
            k_fold,
            num_folds,
            fold=0,
            y_val="global_cog_slope",
            two_pass=True,
            num_nodes=3667,
        )

        # Build train/val sets
        train_features, train_labels, train_sex, train_mutation, train_age = construct_dataset(train)
        val_features, val_labels, val_sex, val_mutation, val_age = construct_dataset(val)

        if use_features:
            X_train, X_val, y_train, y_val = stack_features(
                train_features, train_labels, train_sex, train_mutation, train_age,
                val_features, val_labels, val_sex, val_mutation, val_age,
            )
        else:
            # Only use original train_features, no sex/mutation/age
            X_train, X_val = train_features, val_features
            y_train, y_val = train_labels, val_labels
            print("Using original features")
            print(train_features.shape)
            print(val_features.shape)

        print(f"[k_fold=False] Training on {X_train.shape[0]} samples, validating on {X_val.shape[0]} samples")

        # Tune alpha on val
        alpha_results = {alpha: [] for alpha in alphas}
        alpha_results_orig = {alpha: [] for alpha in alphas}

        best_alpha = None
        best_mse = float("inf")

        for alpha in alphas:
            model = Lasso(alpha=alpha, random_state=42, max_iter=5000)
            model.fit(X_train, y_train)

            preds = model.predict(X_val)
            mse = mean_squared_error(y_val, preds)
            mse_orig = convert_results(mse, adj_metric, adj_thresh, False, num_folds, 0)

            alpha_results[alpha].append(mse)
            alpha_results_orig[alpha].append(mse_orig)

            if mse < best_mse:
                best_mse = mse
                best_alpha = alpha

        # Optional: print alpha summary
        print("\nValidation results (k_fold=False):")
        print("Alpha\tVal MSE\tVal MSE (orig units)")
        print("-" * 40)
        for alpha in alphas:
            if alpha_results[alpha]:
                print(f"{alpha:.4f}\t{alpha_results[alpha][0]:.4f}\t{alpha_results_orig[alpha][0]:.4f}")

        print(f"\nBest alpha on val set: {best_alpha} (val MSE={best_mse:.4f})")

        # Fit best model on TRAIN only, report TRAIN performance
        best_model = Lasso(alpha=best_alpha, random_state=42, max_iter=5000)
        best_model.fit(X_train, y_train)

        train_preds = best_model.predict(X_train)
        train_mse = mean_squared_error(y_train, train_preds)
        train_mse_orig = convert_results(train_mse, adj_metric, adj_thresh, False, num_folds, 0)

        print(f"Final train MSE with best alpha: {train_mse:.4f}")
        print(f"Final train MSE (orig units): {train_mse_orig:.4f}")

        return {
            "mode": "no_kfold",
            "alpha_val_mse": alpha_results,
            "alpha_val_mse_orig": alpha_results_orig,
            "best_alpha": best_alpha,
            "val_best_mse": best_mse,
            "train_mse": train_mse,
            "train_mse_orig": train_mse_orig,
        }


def get_age_sex_mutation_info(num_folds=5, k_fold=True, expand_features=True, adj_metric="pointcloud", adj_thresh=1.0):
    for fold in range(num_folds):
        [train, val, test] = load_dataset(adj_metric, adj_thresh, k_fold, num_folds, fold=fold, y_val="global_cog_slope")
        train_features, train_labels, train_sex, train_mutation, train_age = construct_dataset(train)
        val_features, val_labels, val_sex, val_mutation, val_age = construct_dataset(val)
        return(train_sex, train_mutation, train_age, val_sex, val_mutation, val_age)

In [4]:
run_lasso_folds(num_folds=5, use_features=True)

Processed file names: ['FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_two_pass_True_train.pt', 'FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_two_pass_True_val.pt']
Loading data from: /scratch/lcornelis/data/data_louisa/FTD/processed/FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_two_pass_True_train.pt
Processed file names: ['FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_two_pass_True_train.pt', 'FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_two_pass_True_val.pt']
Loading data from: /scratch/lcornelis/data/data_louisa/FTD/processed/FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_

{'mode': 'cv',
 'alpha_val_mse': {np.float64(0.0001): [np.float64(0.6539149936959048),
   np.float64(0.597608088053605),
   np.float64(0.4617661675652776),
   np.float64(0.48759683328750086),
   np.float64(0.41253459120571573)],
  np.float64(0.00013257113655901095): [np.float64(0.6271389014810186),
   np.float64(0.5892476371036406),
   np.float64(0.4715044430809502),
   np.float64(0.48087621303256584),
   np.float64(0.4553452004210772)],
  np.float64(0.00017575106248547912): [np.float64(0.6207173815211321),
   np.float64(0.574239467183543),
   np.float64(0.4711819633347581),
   np.float64(0.5067914686596421),
   np.float64(0.494926607550086)],
  np.float64(0.00023299518105153718): [np.float64(0.6165918767637307),
   np.float64(0.5773754235340405),
   np.float64(0.47680082897120446),
   np.float64(0.5161082622143683),
   np.float64(0.5197104427831389)],
  np.float64(0.00030888435964774815): [np.float64(0.6257986871244589),
   np.float64(0.6100432166183262),
   np.float64(0.4786687002757

In [8]:
run_lasso_folds(num_folds=1, use_features=True, k_fold=False)

Processed file names: ['FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_two_pass_True_train.pt', 'FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_two_pass_True_val.pt', 'FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_two_pass_True_test.pt']
Loading data from: /scratch/lcornelis/data/data_louisa/FTD/processed/FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_two_pass_True_train.pt
Processed file names: ['FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_two_pass_True_train.pt', 'FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_two_pass_True_val.pt', 'FTD_y_val_

{'mode': 'no_kfold',
 'alpha_val_mse': {np.float64(0.0001): [np.float64(0.7022386548208435)],
  np.float64(0.00013257113655901095): [np.float64(0.6994148714396323)],
  np.float64(0.00017575106248547912): [np.float64(0.7100737672933981)],
  np.float64(0.00023299518105153718): [np.float64(0.7089776840983071)],
  np.float64(0.00030888435964774815): [np.float64(0.6901050199987196)],
  np.float64(0.00040949150623804275): [np.float64(0.6707582931957691)],
  np.float64(0.0005428675439323859): [np.float64(0.6904350943446061)],
  np.float64(0.0007196856730011522): [np.float64(0.6993241755004518)],
  np.float64(0.0009540954763499944): [np.float64(0.7123100736011116)],
  np.float64(0.0012648552168552957): [np.float64(0.7137601881496868)],
  np.float64(0.0016768329368110084): [np.float64(0.7190825749919152)],
  np.float64(0.0022229964825261957): [np.float64(0.7161980487772078)],
  np.float64(0.0029470517025518097): [np.float64(0.7135510498665216)],
  np.float64(0.003906939937054617): [np.float64(0

In [16]:
run_lasso_folds(num_folds=5, expand_features=False)

TypeError: run_lasso_folds() got an unexpected keyword argument 'expand_features'

In [18]:
run_lasso_folds(num_folds=5, expand_features=True)

Processed file names: ['FTD_y_val_nfl_wgcna_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_train.pt', 'FTD_y_val_nfl_wgcna_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_val.pt']
Loading data from: /scratch/lcornelis/data/data_louisa/FTD/processed/FTD_y_val_nfl_wgcna_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_train.pt
Processed file names: ['FTD_y_val_nfl_wgcna_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_train.pt', 'FTD_y_val_nfl_wgcna_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_val.pt']
Loading data from: /scratch/lcornelis/data/data_louisa/FTD/processed/FTD_y_val_nfl_wgcna_adj_thresh_0.5_num_nodes_7258_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_5fold_0_val.pt
Processed file names: ['FTD_y_val_nfl_wgcna_adj_thresh_0.5_n

{np.float64(0.0001): [np.float64(0.7452082209831214),
  np.float64(0.5885696531128094),
  np.float64(1.0671477062486177),
  np.float64(0.6436696235700892),
  np.float64(0.5157570432695622)],
 np.float64(0.00013257113655901095): [np.float64(0.7652010952532736),
  np.float64(0.5626686952823319),
  np.float64(1.0503653268187845),
  np.float64(0.5755241208657731),
  np.float64(0.457871178388135)],
 np.float64(0.00017575106248547912): [np.float64(0.7839078057109096),
  np.float64(0.5328763256730625),
  np.float64(1.0808960675573605),
  np.float64(0.5555989707091538),
  np.float64(0.41533951713565626)],
 np.float64(0.00023299518105153718): [np.float64(0.83152735300404),
  np.float64(0.5108517396834447),
  np.float64(1.1195932357363874),
  np.float64(0.6047431685446067),
  np.float64(0.3937444171833427)],
 np.float64(0.00030888435964774815): [np.float64(0.8644023637718001),
  np.float64(0.5011362733268236),
  np.float64(1.1356248435185814),
  np.float64(0.6321505752752703),
  np.float64(0.395

In [7]:
run_lasso_folds(num_folds=1, expand_features=True, k_fold=False)

Processed file names: ['FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_two_pass_True_train.pt', 'FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_two_pass_True_val.pt']
Loading data from: /scratch/lcornelis/data/data_louisa/FTD/raw/cleanDat.Soma.CSFneat.5SDWinsor_merged.csv


Processing...


Number of patients with measurements: 248.0
Number of patients with mutation status in ['GRN', 'MAPT', 'C9orf72', 'CTL']: 248
Number of patients with sex in ['M', 'F']: 248
Total number of patients with all conditions 248
final dims of filtered data: (246, 4308)
Number of proteins: 3667
Mean and std saved to: /scratch/lcornelis/data/data_louisa/FTD/processed/FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_train_random_state_42.json
Training features and labels: torch.Size([196, 3667, 1]) torch.Size([196])
Val features and labels: torch.Size([50, 3667, 1]) torch.Size([50])
Training sex, mutation and age labels shape: torch.Size([196, 1]) torch.Size([196, 1]) torch.Size([196, 1])
Val sex, mutation and age labels shape: torch.Size([50, 1]) torch.Size([50, 1]) torch.Size([50, 1])
   Power SFT.R.sq   slope truncated.R.sq mean.k. median.k. max.k.
1      1  0.94700  5.6400          0.971  2240.0   2320.00   2640
2      2  0.83300  1.920

Done!


Loading data from: /scratch/lcornelis/data/data_louisa/FTD/processed/FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_two_pass_True_train.pt
Processed file names: ['FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_two_pass_True_train.pt', 'FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_two_pass_True_val.pt']
Loading data from: /scratch/lcornelis/data/data_louisa/FTD/processed/FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_two_pass_True_val.pt
Processed file names: ['FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,CTL_csf_sex_M,F_random_state_42_two_pass_True_train.pt', 'FTD_y_val_global_cog_slope_wgcna_adj_thresh_0.5_num_nodes_3667_mutation_GRN,MAPT,C9orf72,C

{np.float64(0.0001): [np.float64(0.5839507110925711)],
 np.float64(0.00013257113655901095): [np.float64(0.5827489332216288)],
 np.float64(0.00017575106248547912): [np.float64(0.5986146712953615)],
 np.float64(0.00023299518105153718): [np.float64(0.6210462090660814)],
 np.float64(0.00030888435964774815): [np.float64(0.6495314215509634)],
 np.float64(0.00040949150623804275): [np.float64(0.667225801180678)],
 np.float64(0.0005428675439323859): [np.float64(0.6860419047578649)],
 np.float64(0.0007196856730011522): [np.float64(0.6896197387715016)],
 np.float64(0.0009540954763499944): [np.float64(0.7058092540827207)],
 np.float64(0.0012648552168552957): [np.float64(0.7004502832018789)],
 np.float64(0.0016768329368110084): [np.float64(0.6875896770725173)],
 np.float64(0.0022229964825261957): [np.float64(0.6909297378656812)],
 np.float64(0.0029470517025518097): [np.float64(0.703230622196362)],
 np.float64(0.003906939937054617): [np.float64(0.7079746038555935)],
 np.float64(0.005179474679231213)

In [20]:
During the k-fold -> 143 training samples, validating on 36 samples
Final evaluation -> 179 total training samples, testing on 45
7258 proteins

SyntaxError: invalid syntax (903581591.py, line 1)